### Solving an inverse Problem with Qewton and PINNs
Assume you are jumping from a plane with a parachute. Under some simplifications, your fall can be described by the following ODE:
\begin{align*}
    \begin{split}
        \partial_t^2 u(t) &= {\color{red}{D(t)}} \Big[\partial_t u(t)\Big]^2 - g  \quad 
        \text{ for } t\in (0,T), \\
        u(0) &= H, \\
        \partial_t u(0) &= v_0,
    \end{split}
\end{align*}
where $u : [0, T) \to \mathbb{R}$ is the height at time $t$, $H$ the starting height, $g$ the gravity constant, $v_0$ the initial velocity, and $\color{red}{D(t)}$ a friction coefficient at time $t$.

Given a noisy dataset $\{(u_i, t_i)\}_{i=1}^N$ of the height $u_i$ at time $t_i$, we aim to reconstruct the movement of the parachute described by $\color{red}{D(t)}$.

In [12]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [13]:
import qewton

path = "/localdata/tomfre/SolutionData/ParachuteData/"

# Here all parameters are defined:
g = 9.81                # Gravity constant
H = 0                   # Initial height H
v_0 = -2                # Initial velocity v_0
t_min, t_max = 0.0, 10 # Bounds for time interval

### Spaces, domains and samplers

In [14]:
T = qewton.Variable('t', dim=1)
U = qewton.Variable('u', dim=1)
# TODO: Create the output space of parameter function D
D = qewton.Variable('D', dim=1)

# TODO: Define the time interval I_t
I_t = qewton.geometries.Interval(T, t_min, t_max)

# TODO: Create random uniform sampler for the ODE condition, which should hold on I_t.
# Use 3000 as the number of sampling points
ode_sampler = qewton.GridSampler(I_t, 5000)

In [15]:
# Create the models that will be used to learn the solution u(t) and 
# the parameter function D(t)
model_u = qewton.FCN(
    in_neurons=T,
    hidden_neurons=16,
    out_neurons=U,
    n_hidden_layers=2,
    activation=qewton.bb.Tanh,
    )

# TODO: Create a FCN for the parameter function D(t). Use hidden=16
model_D = qewton.FCN(
    in_neurons=T,
    hidden_neurons=16,
    out_neurons=D,
    n_hidden_layers=2,
    activation=qewton.bb.Tanh,
    )

In [16]:
# Load data
t_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "time_data.pt")
u_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "noisy_measurement.pt")

In [17]:
# Create the data loader for the measurement data
data_size = len(u_data)
input_config = qewton.config.DataConfiguration(
    qewton.config.BatchAxes(data_size), qewton.config.FeatureAxes(T)
)
output_config = qewton.config.DataConfiguration(
    qewton.config.BatchAxes(data_size), qewton.config.FeatureAxes(U)
)

dataset = qewton.data.ArrayLikeDataSet(
    data=[t_data, u_data], data_configs=[input_config, output_config]
)

data_loader = qewton.data.DataLoader(
    data_set=dataset,
    batch_size=data_size,
    shuffle_data=False,
)

In [18]:
plot_config = qewton.config.DataConfiguration(
    qewton.config.GeometryAxes(shape=(data_size,)), qewton.config.FeatureAxes(U)
)
data_plot = qewton.viz.LinePlot(u_data, plot_config, T, U)
qewton.viz.Figure(data_plot, title="Measurement data of u").show()

In [19]:
# Now we build the computation graph for the data constraint. 
# The data constraint is a mean squared error (MSE) constraint that compares 
# the model's predictions with the actual measurements:
data_constraint = qewton.constraints.MSEConstraint(weight=10.0)
data_pipeline = qewton.Graph()

with data_pipeline.tracker(0):
    t, u = data_loader()
    model_prediction = model_u(t)
    data_constraint(model_prediction, u)

In [20]:
# TODO: Create a residual for the ODE condition
def ode_residual(t : T, u: U, D: D):
  u_t = u.gradient(t)
  u_tt = u_t.gradient(t)
  return u_tt - D*u_t**2 + g

ode_constraint = qewton.PINNConstraint(
  ode_residual, 
  activation_condition=qewton.CosineRampWeight(2000, 4000),
  name="ODEConstraint")

ode_pipeline = qewton.PINNPipeline(
    sampler=ode_sampler,
    constraint=ode_constraint,
    models=[model_u, model_D]
)

In [21]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=10000
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=1.0,
    max_iterations=500,
    optimizer_args={"max_eval": 10}
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=[ode_pipeline, data_pipeline],
    training_objectives=[ode_constraint, data_constraint],
    device=qewton.cuda(0)
)

trainer.run()

Optimization Phase 1:   0%|          | 9/10000 [00:00<01:29, 111.73it/s, loss=3.54e+3]

Optimization Phase 2: 100%|██████████| 500/500 [00:47<00:00, 10.63it/s, loss=0.00893]


In [22]:
# Now we can check how good the D can be reconstructed. 
# We can plot the learned D(t) and compare it with the true 
# D(t) = 3 * (1.1 + sin(-0.42*t))
import numpy as np

model_D.to(qewton.cpu)

def true_D(t):
    return 3 * (1.1 + np.sin(-0.42*t))

layout = ode_pipeline.visualize(model_D.output_ports[0], reference=true_D)
qewton.viz.Figure(layout).show()